# Finding the Manifold (Physics-Informed Neural Networks) and Hybrid Models
## 3. The "Gray Box"

### 🧠 The Deep Learning Problem in BiologyWhy not just throw a standard Deep Neural Network (DNN) at your bioreactor data?

If you build a standard DNN (e.g., using PyTorch or TensorFlow) to predict mAb titer based on time, temperature, and pH, the network acts as a universal function approximator. However, DNNs are notoriously data-hungry. In tech, they are trained on millions of images. In a CRO, you might only have data from 50 Ambr15 microbioreactor runs.\
If you train a pure DNN on 50 runs, it will fiercely overfit. Even worse, because it knows no biology, it might predict physically impossible phenomena—like the glucose concentration dropping below $0$ g/L, or cell mass spontaneously generating without consuming any food.We need to constrain the neural network.\
We need to force it to obey the laws of physics.
### ⚖️ The PINN Paradigm: Constraining the Manifold
A Physics-Informed Neural Network (PINN) is fundamentally just a standard neural network, but with a highly specialized Loss Function.\
In standard Deep Learning (like your Phase 1 model), your loss function was just the Mean Squared Error (MSE) between the prediction and the data:$$\mathcal{L}_{Data} = \frac{1}{N} \sum (y_{true} - y_{pred})^2$$In a PINN, you add a massive penalty to the Loss Function if the network's predictions violate the ODEs (the biological laws). We call this the Physics Loss:$$\mathcal{L}_{Total} = \mathcal{L}_{Data} + \lambda \mathcal{L}_{Physics}$$(Where $\lambda$ is a weight you choose to balance the two).

How does $\mathcal{L}_{Physics}$ work?\
Remember the cell growth ODE from Chapter 3?\
$\frac{dX}{dt} = \mu X$\
If we rearrange this, it equals zero:\
$\frac{dX}{dt} - \mu X = 0$\
During training, the neural network uses automatic differentiation (autograd in PyTorch) to calculate the derivative of its own predictions ($\frac{dX}{dt}$). It then plugs that into the ODE. If the math doesn't equal $0$, the network is breaking the laws of biology! The $\mathcal{L}_{Physics}$ penalty spikes, and the backpropagation algorithm forces the network's weights to adjust until it obeys the ODE.
- The Metaphor: Standard ML is a teenager trying to learn how to drive by only looking at a few photos of steering wheels. A PINN is that same teenager, but you have placed guardrails on the physical road and handed them the physics equations for momentum and friction. They learn infinitely faster and never drive off the cliff.

### 🧬 The "Grey Box" Hybrid Model (The Industry Standard)
While pure PINNs are incredibly powerful, they can be difficult to train. In the biopharma industry, we often use a slightly different architecture called a **Hybrid Semi-Parametric Model**. This is the ultimate tool for finding your bioprocess manifold.\
Here is how it works:
- The Biological Anchor (White Box): We keep the odeint solver. We trust the Monod and Luedeking-Piret equations.
- The Machine Learning Brain (Black Box): We don't know exactly how Temperature, pH, and Dissolved Oxygen (DO) dynamically alter the cell's maximum growth rate ($\mu_{max}$) or its mAb secretion rate ($\beta$). So, we use a Neural Network to predict those specific parameters.
**The Architecture:** Instead of the Neural Network predicting the final Titer directly, it predicts the coefficients of the ODEs based on the environment!/
  $$Inputs: [Temp, pH, DO] \xrightarrow{Deep Neural Network} Outputs: [\mu_{max}, K_s, \alpha, \beta]$$\
  Then, those ML-predicted parameters are fed instantly into the ODE solver to generate the 14-day curves.

  ## 💻 Exercise: Conceptualizing the Hybrid PINN in PyTorch
  This is a PSEUDO-CODE rather than executable

In [ ]:
###   This is a PSEUDO-CODE rather than executable  ###

import torch
import torch.nn as nn
# (Assume a differentiable ODE solver library like torchdiffeq is imported)

# 1. Define the Black Box (The Neural Network)
class ParameterPredictior(nn.Module):
    def __inti__(self):
        super().__init__()
        # A simple feed-forward network
        self.net = nn.Sequential(nn.Lienar(3, 16), # Inputs: Temp, pH, DO
                                 nn.ReLU(),
                                 nn.Linear(16, 4) # Outputs: mu_max, Ks, alpha, beta
                                )

    def forward(self, environment_inputs):
        # Force outputs to be positive using absolute value or Softplus
        # Biology constraint: you can't have a negative growth rate!
        return torch.nn.fucntional.softplus(self.net(environment_inputs))

# 2. Define the White Box (The Biological ODEs)
def biological_ode(t, y, predicted_params):
    # This is exactly the Ordinary Differenial Equations math, but written in PyTorch tensors!
    X, S, P = y[0], y[1], y[2]
    mu_max, Ks, alpha, beta = predicted_params

    # Equations:
    mu = mu_max * (S / (Ks + S))
    dXdt = mu * X
    dSdt = -(1/0.4) * dXdt # Assuming fixed yield for simplicity
    dPdt = alpha * dXdt + beta * X

    return torch.stack([dXdt, dSdt, dPdt])

# 3. The "Grey Box" Training Loop (The Magic)
for every epoch in traininig:
    # A. Pass the DoE inputs (Temp, pH) into the Neural Network
    params = model(environmental_inputs)

    # B. Pass the predicted params into the ODE solver to get 14-day predictions
    predicted_curves = odeint(biological_ode, initial_conditions, time, args=(params,))

    # C. Calculate the Loss (MSE between the ODE output and actual Bioreactor data)
    loss = torch.nn.MSELoss()(predicted_curves, actual_bioreactor_data)

    # D. Backpropagate the error THROUGH the ODE solver, back into the Neural Network!
    loss.backward()
    optimizer.step()